In [13]:
import pandas as pd
import numpy as np
3+5

8

In [14]:
iv_char_switch = pd.read_csv('outputs/CREE_C3M0016120K/1_iv_characteristics_switch.csv')
iv_char_switch

,temperature_C,vgs_V,vds_V,id_A,rds_on_Ohm,power_W
0,-40,7,0.000000,0.000000,NaN,0.000000
1,-40,7,0.445692,2.659503,0.167585,1.185320
2,-40,7,1.009426,6.496544,0.155379,6.557783
3,-40,7,1.630825,8.891281,0.183418,14.500128
4,-40,7,2.390328,11.619294,0.205721,27.773927
...,...,...,...,...,...,...
311,175,15,9.190000,221.020000,0.041580,2031.173800
312,175,15,9.750000,228.280000,0.042711,2225.730000
313,175,15,10.520000,237.370000,0.044319,2497.132400
314,175,15,11.240000,243.920000,0.046081,2741.660800


In [15]:
iv_char_diode = pd.read_csv('outputs/CREE_C3M0016120K/2_iv_characteristics_diode.csv')
iv_char_diode

,temperature_C,vgs_V,vds_V,id_A,rds_on_Ohm,power_W
0,25,-4,0.000000,0.000000,NaN,0.000000
1,25,-4,2.745425,0.000000,NaN,0.000000
2,25,-4,3.245867,5.367817,0.604690,17.423221
3,25,-4,3.691040,13.229283,0.279005,48.829811
4,25,-4,4.136656,25.885255,0.159807,107.078399
...,...,...,...,...,...,...
74,175,0,3.988764,75.452793,0.052864,300.963417
75,175,0,4.420718,99.623882,0.044374,440.409070
76,175,0,5.172621,149.006434,0.034714,770.753845
77,175,0,5.812521,199.429359,0.029146,1159.187408


In [ ]:
cap_volt = pd.read_csv('outputs/CREE_C3M0016120K/3_capacitance_voltage.csv')
cap_volt
# only includes 1 temperature

,temperature_C,voltage_V,c_iss_F,c_oss_F,c_rss_F
0,25,0.000000,7.677300e-09,6.570600e-09,2.452700e-09
1,25,0.964630,NaN,NaN,1.250200e-09
2,25,1.607700,NaN,4.692300e-09,NaN
3,25,2.250800,NaN,NaN,9.649900e-10
4,25,2.572300,6.575000e-09,NaN,NaN
...,...,...,...,...,...
158,25,927.835052,NaN,2.231967e-10,NaN
159,25,1057.731959,NaN,NaN,1.269753e-11
160,25,1068.041237,NaN,2.221227e-10,NaN
161,25,1193.814433,NaN,2.211637e-10,1.263822e-11


In [17]:
# EDA and analysis will include derived 

derived = pd.read_csv('outputs/CREE_C3M0016120K/4_derived_parameters.csv')
derived

,temperature_C,vgs_V,vth_estimate_V,lambda_per_V,gds_S,gm_estimate_S,id_sat_A
0,-40,7,NaN,NaN,NaN,NaN,21.369260
1,-40,9,NaN,6.408031,35.912152,NaN,5.604241
2,-40,11,NaN,1.962961,35.748269,NaN,18.211397
3,-40,13,NaN,0.788392,26.634330,NaN,33.783095
4,-40,15,NaN,0.515547,23.459143,NaN,45.503413
5,25,7,NaN,NaN,NaN,NaN,47.990000
6,25,9,NaN,5.085714,46.809335,NaN,9.204083
7,25,11,NaN,1.040350,34.882876,NaN,33.529945
8,25,13,NaN,2.263075,47.050515,NaN,20.790526
9,25,15,NaN,0.813711,33.222236,NaN,40.828073


In [18]:
thermal_voltage = pd.read_csv('outputs/CREE_C3M0016120K/5_thermal_voltage.csv')
thermal_voltage

,temperature_C,temperature_K,thermal_voltage_V
0,-40,233.15,0.020091
1,25,298.15,0.025693
2,175,448.15,0.038619


# Performing Analysis

In [22]:
# Calculating Normalized sensitivities


def compute_normalized_sensitivity(df, col, group_col=None):
    if group_col:
        mean_by_temp = df.groupby('temperature_C')[col].mean()
    else:
        mean_by_temp = df.set_index('temperature_C')[col]
     
    mean_val = mean_by_temp.mean()
    delta_val = mean_by_temp.max() - mean_by_temp.min()
    delta_temp = mean_by_temp.index.max() - mean_by_temp.index.min()
    
    if mean_val == 0 or delta_temp == 0:
        return np.nan  # Avoid division by zero
    
    return (delta_val / mean_val) / delta_temp

In [23]:
# call for each parameter in teh dfs:

parameters = {
    # iv_char_switch
    'Id_switch': (iv_char_switch, 'id_A', 'vgs_V'),
    'Rds_on_switch': (iv_char_switch, 'rds_on_Ohm', 'vgs_V'),

    # iv_char_diode
    'Id_diode': (iv_char_diode, 'id_A', 'vgs_V'),
    'Rds_on_diode': (iv_char_diode, 'rds_on_Ohm', 'vgs_V'),  # optional

    # cap_volt
    'Ciss': (cap_volt, 'c_iss_F', 'voltage_V'),
    'Coss': (cap_volt, 'c_oss_F', 'voltage_V'),
    'Crss': (cap_volt, 'c_rss_F', 'voltage_V'),

    # derived - take with a grain of salt, derivations not confirmed yet
    'Vth': (derived, 'vth_estimate_V', 'vgs_V'),
    'gm': (derived, 'gm_estimate_S', 'vgs_V'),
    'gds': (derived, 'gds_S', 'vgs_V'),
    'Id_sat': (derived, 'id_sat_A', 'vgs_V'),

    # thermal_voltage - ignore idk, there was only 1 value for each temp, small sample size
    'Vt_thermal': (thermal_voltage, 'thermal_voltage_V', None)
}

In [31]:
sensitivity_results = {}
for param, (df, col, group) in parameters.items():
    sensitivity_results[param] = compute_normalized_sensitivity(df, col, group)

# Convert to a sorted DataFrame for reporting
sens_df = pd.DataFrame.from_dict(sensitivity_results, orient='index', columns=['Normalized Sensitivity'])
sens_df = sens_df.sort_values('Normalized Sensitivity', ascending=False)
srdf = pd.DataFrame([sensitivity_results])
srdf

,Id_switch,Rds_on_switch,Id_diode,Rds_on_diode,Ciss,Coss,Crss,Vth,gm,gds,Id_sat,Vt_thermal
0,0.000789,0.002041,0.000081,0.000014,NaN,NaN,NaN,NaN,NaN,0.00141,0.003193,0.003063
